## 01_luna_manifest_y_subsets.ipynb
 Objetivo:
 1) detectar automáticamente los subsets disponibles de LUNA16
 2) localizar scans .mhd y verificar su archivo de datos real
 3) leer annotations.csv y candidates_V2.csv
 4) construir un manifest maestro por volumen
 5) generar split train/val/test reproducible

## imports

In [2]:
from pathlib import Path
import re
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display

## configuración de rutas

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/mnt/d/Universidad/analitica/proyecto_analitica2")
DATA_DIR = PROJECT_ROOT / "data"
SOURCE_DIR = DATA_DIR / "source"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"

LUNA16_SOURCE_DIR = SOURCE_DIR / "luna16"
LUNA16_WORKING_DIR = WORKING_DIR / "luna16"

MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
LUNA16_WORKING_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LUNA16_SOURCE_DIR:", LUNA16_SOURCE_DIR)
print("LUNA16_WORKING_DIR:", LUNA16_WORKING_DIR)
print("MANIFESTS_DIR:", MANIFESTS_DIR)
print("LUNA16_SOURCE_DIR existe:", LUNA16_SOURCE_DIR.exists())

PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
LUNA16_SOURCE_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/luna16
LUNA16_WORKING_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working/luna16
MANIFESTS_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests
LUNA16_SOURCE_DIR existe: True


## inspección rápida de la carpeta

In [4]:
items = sorted([p.name for p in LUNA16_SOURCE_DIR.iterdir()])
print("Contenido actual de luna16/:")
for x in items:
    print(" -", x)

Contenido actual de luna16/:
 - 3723295.zip
 - annotations.csv
 - candidates.csv
 - candidates_V2.csv
 - candidates_V2.zip
 - evaluationScript.zip
 - sampleSubmission.csv
 - seg-lungs-LUNA16.zip
 - subset0
 - subset0.zip
 - subset1
 - subset1.zip
 - subset2
 - subset2.zip
 - subset3
 - subset3.zip
 - subset4
 - subset4.zip
 - subset5
 - subset5.zip
 - subset6
 - subset6.zip


## detectar subsets realmente disponibles

In [5]:
subset_pattern = re.compile(r"subset\d+$")

subset_dirs = sorted(
    [p for p in LUNA16_SOURCE_DIR.iterdir() if p.is_dir() and subset_pattern.fullmatch(p.name)],
    key=lambda p: int(p.name.replace("subset", ""))
)

print("Subsets detectados:")
for p in subset_dirs:
    print(" -", p.name)

print("\nCantidad de subsets detectados:", len(subset_dirs))

Subsets detectados:
 - subset0
 - subset1
 - subset2
 - subset3
 - subset4
 - subset5
 - subset6

Cantidad de subsets detectados: 7


## helper para leer el archivo de datos real del .mhd

In [6]:
def get_element_data_file(mhd_path: Path):
    with open(mhd_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line.startswith("ElementDataFile"):
                return line.split("=")[1].strip()
    return None

## localizar todos los scans .mhd de los subsets válidos

In [7]:
mhd_files = []
for subset_dir in subset_dirs:
    mhd_files.extend(sorted(subset_dir.glob("*.mhd")))

print(f"Scans .mhd encontrados en subsets válidos: {len(mhd_files)}")
mhd_files[:5]

Scans .mhd encontrados en subsets válidos: 623


[PosixPath('/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/luna16/subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.mhd'),
 PosixPath('/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/luna16/subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896446896160048741492.mhd'),
 PosixPath('/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/luna16/subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524522225658609808059.mhd'),
 PosixPath('/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/luna16/subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.111172165674661221381920536987.mhd'),
 PosixPath('/mnt/d/Universidad/analitica/proyecto_analitica2/data/source/luna16/subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896761494371822656720.mhd')]

## construir tabla base de scans

In [8]:
rows = []

for mhd_path in mhd_files:
    seriesuid = mhd_path.stem
    subset_name = mhd_path.parent.name
    element_data_file = get_element_data_file(mhd_path)

    if element_data_file is not None:
        data_path = mhd_path.parent / element_data_file
        data_exists = data_path.exists()
    else:
        data_path = None
        data_exists = False

    rows.append({
        "seriesuid": seriesuid,
        "subset": subset_name,
        "mhd_path": str(mhd_path.resolve()),
        "data_file_name": element_data_file,
        "data_path": str(data_path.resolve()) if data_path is not None else None,
        "data_exists": data_exists,
    })

df_scans = pd.DataFrame(rows)

print("Shape df_scans:", df_scans.shape)
display(df_scans.head())

Shape df_scans: (623, 6)


,seriesuid,subset,mhd_path,data_file_name,data_path,data_exists
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031...,/mnt/d/Universidad/analitica/proyecto_analitic...,True
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896...,/mnt/d/Universidad/analitica/proyecto_analitic...,True
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524...,/mnt/d/Universidad/analitica/proyecto_analitic...,True
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.111172165674...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.111172165674...,/mnt/d/Universidad/analitica/proyecto_analitic...,True
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896...,/mnt/d/Universidad/analitica/proyecto_analitic...,True


## chequeo de integridad 


In [9]:
print("Total CT scans encontrados:", len(df_scans))
print("Archivos de datos faltantes:", (~df_scans["data_exists"]).sum())

print("\nScans por subset:")
print(df_scans["subset"].value_counts().sort_index())

Total CT scans encontrados: 623
Archivos de datos faltantes: 0

Scans por subset:
subset
subset0    89
subset1    89
subset2    89
subset3    89
subset4    89
subset5    89
subset6    89
Name: count, dtype: int64


## cargar annotations.csv

In [10]:
ann_path = LUNA16_SOURCE_DIR / "annotations.csv"

if ann_path.exists():
    df_ann = pd.read_csv(ann_path)
    print("annotations.csv cargado:", df_ann.shape)
    display(df_ann.head())
else:
    df_ann = None
    print("No se encontró annotations.csv")

annotations.csv cargado: (1186, 5)


,seriesuid,coordX,coordY,coordZ,diameter_mm
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-128.699421,-175.319272,-298.387506,5.651471
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,103.783651,-211.925149,-227.121250,4.224708
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,69.639017,-140.944586,876.374496,5.786348
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,-24.013824,192.102405,-391.081276,8.143262
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,2.441547,172.464881,-405.493732,18.545150


## cargar candidatos, prefiriendo candidates_V2.csv

In [11]:
cand_v2_path = LUNA16_SOURCE_DIR / "candidates_V2.csv"
cand_path = LUNA16_SOURCE_DIR / "candidates.csv"

if cand_v2_path.exists():
    df_cand = pd.read_csv(cand_v2_path)
    cand_source = "candidates_V2.csv"
elif cand_path.exists():
    df_cand = pd.read_csv(cand_path)
    cand_source = "candidates.csv"
else:
    df_cand = None
    cand_source = None

print("Fuente de candidatos usada:", cand_source)

if df_cand is not None:
    print("Shape candidatos:", df_cand.shape)
    display(df_cand.head())
else:
    print("No se encontró archivo de candidatos")

Fuente de candidatos usada: candidates_V2.csv
Shape candidatos: (754975, 5)


,seriesuid,coordX,coordY,coordZ,class
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,68.420000,-74.480000,-288.700000,0
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-95.209361,-91.809406,-377.426350,0
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-24.766755,-120.379294,-273.361539,0
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-63.080000,-65.740000,-344.240000,0
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,52.946688,-92.688873,-241.067872,0


## resumen por scan usando anotaciones

In [12]:
if df_ann is not None and "seriesuid" in df_ann.columns:
    ann_summary = (
        df_ann.groupby("seriesuid")
        .size()
        .reset_index(name="n_annotations")
    )
    ann_summary["has_annotation"] = 1
else:
    ann_summary = pd.DataFrame(columns=["seriesuid", "n_annotations", "has_annotation"])

print("Shape ann_summary:", ann_summary.shape)
display(ann_summary.head())

Shape ann_summary: (601, 3)


,seriesuid,n_annotations,has_annotation
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,2,1
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,1,1
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,4,1
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100953483028...,1,1
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.102681962408...,1,1


## resumen por scan usando candidatos

In [13]:
if df_cand is not None and {"seriesuid", "class"}.issubset(df_cand.columns):
    cand_summary = (
        df_cand.groupby("seriesuid")
        .agg(
            n_candidates=("class", "size"),
            n_positive_candidates=("class", lambda x: int((x == 1).sum())),
            n_negative_candidates=("class", lambda x: int((x == 0).sum())),
        )
        .reset_index()
    )
else:
    cand_summary = pd.DataFrame(
        columns=[
            "seriesuid",
            "n_candidates",
            "n_positive_candidates",
            "n_negative_candidates",
        ]
    )

print("Shape cand_summary:", cand_summary.shape)
display(cand_summary.head())

Shape cand_summary: (888, 4)


,seriesuid,n_candidates,n_positive_candidates,n_negative_candidates
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,1068,2,1066
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100332161840...,496,0,496
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,1135,1,1134
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100530488926...,472,0,472
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100620385482...,434,0,434


## construir manifest maestro

In [14]:
manifest = df_scans.copy()

if len(ann_summary) > 0:
    manifest = manifest.merge(ann_summary, on="seriesuid", how="left")
else:
    manifest["n_annotations"] = np.nan
    manifest["has_annotation"] = np.nan

if len(cand_summary) > 0:
    manifest = manifest.merge(cand_summary, on="seriesuid", how="left")
else:
    manifest["n_candidates"] = np.nan
    manifest["n_positive_candidates"] = np.nan
    manifest["n_negative_candidates"] = np.nan

for col in [
    "n_annotations",
    "has_annotation",
    "n_candidates",
    "n_positive_candidates",
    "n_negative_candidates",
]:
    manifest[col] = manifest[col].fillna(0).astype(int)

print("Shape manifest:", manifest.shape)
display(manifest.head())

Shape manifest: (623, 11)


,seriesuid,subset,mhd_path,data_file_name,data_path,data_exists,n_annotations,has_annotation,n_candidates,n_positive_candidates,n_negative_candidates
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,0,0,707,0,707
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,1,1,1291,1,1290
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,2,1,804,2,802
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.111172165674...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.111172165674...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,1,1,730,1,729
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,0,0,821,0,821


## crear dos etiquetas binarias y compararlas

In [15]:
manifest["label_from_annotations"] = (manifest["has_annotation"] > 0).astype(int)
manifest["label_from_candidates"] = (manifest["n_positive_candidates"] > 0).astype(int)

print("Distribución label_from_annotations:")
print(manifest["label_from_annotations"].value_counts(dropna=False))

print("\nDistribución label_from_candidates:")
print(manifest["label_from_candidates"].value_counts(dropna=False))

disagreement = (manifest["label_from_annotations"] != manifest["label_from_candidates"]).sum()
print("\nVolúmenes con desacuerdo entre ambas etiquetas:", disagreement)

Distribución label_from_annotations:
label_from_annotations
1    428
0    195
Name: count, dtype: int64

Distribución label_from_candidates:
label_from_candidates
1    428
0    195
Name: count, dtype: int64

Volúmenes con desacuerdo entre ambas etiquetas: 0


## elegir la etiqueta final del baseline inicial

In [16]:
manifest["label_volume_binary"] = manifest["label_from_annotations"].astype(int)

display(
    manifest[
        [
            "seriesuid",
            "subset",
            "label_volume_binary",
            "n_annotations",
            "n_candidates",
            "n_positive_candidates",
        ]
    ].head()
)

,seriesuid,subset,label_volume_binary,n_annotations,n_candidates,n_positive_candidates
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031...,subset0,0,0,707,0
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896...,subset0,1,1,1291,1
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524...,subset0,1,2,804,2
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.111172165674...,subset0,1,1,730,1
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896...,subset0,0,0,821,0


## filtros finales de integridad

In [17]:
manifest = manifest[manifest["data_exists"]].copy()
manifest = manifest.drop_duplicates(subset=["seriesuid"]).reset_index(drop=True)

print("Manifest final tras filtros:", manifest.shape)
print("\nDistribución final de clases:")
print(manifest["label_volume_binary"].value_counts())

Manifest final tras filtros: (623, 14)

Distribución final de clases:
label_volume_binary
1    428
0    195
Name: count, dtype: int64


## split train/val/test estratificado

In [18]:
train_df, temp_df = train_test_split(
    manifest,
    test_size=0.30,
    random_state=SEED,
    stratify=manifest["label_volume_binary"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_volume_binary"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_df = pd.concat([train_df, val_df, test_df], axis=0).reset_index(drop=True)

print("Conteo por split:")
print(split_df["split"].value_counts())

print("\nDistribución por split y clase:")
print(pd.crosstab(split_df["split"], split_df["label_volume_binary"]))

Conteo por split:
split
train    436
test      94
val       93
Name: count, dtype: int64

Distribución por split y clase:
label_volume_binary    0    1
split                        
test                  30   64
train                136  300
val                   29   64


## guardar manifest y split

In [19]:
manifest_path = MANIFESTS_DIR / "manifest_luna16.csv"
split_path = MANIFESTS_DIR / "split_final_luna16.csv"

manifest.to_csv(manifest_path, index=False)
split_df.to_csv(split_path, index=False)

print("Manifest guardado en:", manifest_path)
print("Split guardado en:", split_path)

Manifest guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/manifest_luna16.csv
Split guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/split_final_luna16.csv


## chequeo final

In [20]:
print("Columnas del manifest:")
print(list(manifest.columns))

print("\nColumnas del split:")
print(list(split_df.columns))

display(split_df.head())
display(split_df.sample(min(5, len(split_df)), random_state=SEED))

Columnas del manifest:
['seriesuid', 'subset', 'mhd_path', 'data_file_name', 'data_path', 'data_exists', 'n_annotations', 'has_annotation', 'n_candidates', 'n_positive_candidates', 'n_negative_candidates', 'label_from_annotations', 'label_from_candidates', 'label_volume_binary']

Columnas del split:
['seriesuid', 'subset', 'mhd_path', 'data_file_name', 'data_path', 'data_exists', 'n_annotations', 'has_annotation', 'n_candidates', 'n_positive_candidates', 'n_negative_candidates', 'label_from_annotations', 'label_from_candidates', 'label_volume_binary', 'split']


,seriesuid,subset,mhd_path,data_file_name,data_path,data_exists,n_annotations,has_annotation,n_candidates,n_positive_candidates,n_negative_candidates,label_from_annotations,label_from_candidates,label_volume_binary,split
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.216526102138...,subset2,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.216526102138...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,0,0,501,0,501,0,0,0,train
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.395623571499...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.395623571499...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,1,1,986,1,985,1,1,1,train
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.401389720232...,subset4,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.401389720232...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,1,1,1045,1,1044,1,1,1,train
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.174935793360...,subset5,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.174935793360...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,0,0,488,0,488,0,0,0,train
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.281489753704...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.281489753704...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,1,1,883,1,882,1,1,1,train


,seriesuid,subset,mhd_path,data_file_name,data_path,data_exists,n_annotations,has_annotation,n_candidates,n_positive_candidates,n_negative_candidates,label_from_annotations,label_from_candidates,label_volume_binary,split
249,1.3.6.1.4.1.14519.5.2.1.6279.6001.943403138251...,subset2,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.943403138251...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,1,1,899,3,896,1,1,1,train
558,1.3.6.1.4.1.14519.5.2.1.6279.6001.156016499715...,subset6,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.156016499715...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,3,1,983,3,980,1,1,1,test
174,1.3.6.1.4.1.14519.5.2.1.6279.6001.566816709786...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.566816709786...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,1,1,1091,1,1090,1,1,1,train
280,1.3.6.1.4.1.14519.5.2.1.6279.6001.218476624578...,subset4,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.218476624578...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,2,1,1135,2,1133,1,1,1,train
110,1.3.6.1.4.1.14519.5.2.1.6279.6001.564534197011...,subset0,/mnt/d/Universidad/analitica/proyecto_analitic...,1.3.6.1.4.1.14519.5.2.1.6279.6001.564534197011...,/mnt/d/Universidad/analitica/proyecto_analitic...,True,0,0,753,0,753,0,0,0,train


## Guardar un resumen por subset

In [21]:
subset_summary = (
    manifest.groupby("subset")
    .agg(
        n_scans=("seriesuid", "count"),
        n_positive=("label_volume_binary", "sum"),
    )
    .reset_index()
)

subset_summary["n_negative"] = subset_summary["n_scans"] - subset_summary["n_positive"]
subset_summary["positive_ratio"] = subset_summary["n_positive"] / subset_summary["n_scans"]

display(subset_summary)

,subset,n_scans,n_positive,n_negative,positive_ratio
0,subset0,89,67,22,0.752809
1,subset1,89,61,28,0.685393
2,subset2,89,56,33,0.629213
3,subset3,89,65,24,0.730337
4,subset4,89,62,27,0.696629
5,subset5,89,54,35,0.606742
6,subset6,89,63,26,0.707865


## Guardar ese resumen como CSV

In [22]:
subset_summary_path = MANIFESTS_DIR / "subset_summary_luna16.csv"
subset_summary.to_csv(subset_summary_path, index=False)
print("Resumen por subset guardado en:", subset_summary_path)

Resumen por subset guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/subset_summary_luna16.csv


## Dejar explícito que esta versión usa subsets 0–6

In [23]:
print("Versión actual de LUNA16 usada en este notebook:")
print(" - subsets incluidos:", sorted(manifest['subset'].unique()))
print(" - n_scans:", len(manifest))
print(" - positivos:", int(manifest['label_volume_binary'].sum()))
print(" - negativos:", int((manifest['label_volume_binary'] == 0).sum()))

Versión actual de LUNA16 usada en este notebook:
 - subsets incluidos: ['subset0', 'subset1', 'subset2', 'subset3', 'subset4', 'subset5', 'subset6']
 - n_scans: 623
 - positivos: 428
 - negativos: 195
